# Gradient Diagnostics — Sliced-Wasserstein

We saw two worrying cells in the SW table: **f0 weak under tKSA** and **onset
time sub-chance under fKSA**. Gradient *accuracy* (CGA/FGA) alone cannot tell a
**bug** from a genuinely **uninformative loss landscape** — for that we need a
reference. This notebook triangulates three of them:

1. **FD gradient check** — analytic (autograd) vs numerical gradient. Tests
   whether the *backward* is correct. *Caveat:* only meaningful for smooth
   forwards; for `STE`+`time` it diverges **by design**, and for `time` the
   landscape is multi-scale so the numerical gradient is itself step-size
   dependent (which is a finding, not a pass/fail).
2. **Oracle forward fidelity** — compare each synth's rendered audio to the
   NumPy oracle (onset localization + spectral SNR). Tests whether the
   *forward* is faithful.
3. **Loss-landscape curves** — plot `L(θ)` over the full range (coarse) and
   zoomed (fine). Tests whether the *coarse* trend guides optimization to the
   target, independent of fine-scale ripple.

Grid: both KS synths (`tKSA`/`fKSA`) × all three excitations
(`STE`/`tEXC`/`fEXC`), parameters `{f0, time}`, with smooth controls
(`decay`, `dynamic_level`) to show the checks work.


In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "src").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.synths.synth import Synth, SynthConfig
from src.synths.ddsp import Implementation, ExcitationMode
from src.losses import SlicedWassersteinSpectralLoss

FS = 16000
NUM_SAMPLES = FS * 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

GT = {"f0": 110.0, "time": 0.5, "a1": 0.2, "decay": 0.99,
      "pluck_position": 0.25, "burst_gain": 0.9, "dynamic_level": 0.9}

PARAM_RANGES = {"f0": (55.0, 880.0), "time": (0.01, 0.99),
                "decay": (0.9, 0.999), "dynamic_level": (0.5, 1.0)}

# Full grid of synths (single-event LTI setting, matching SW Table 1).
KS = {"tKSA": Implementation.TIME_DOMAIN, "fKSA": Implementation.FREQUENCY_SAMPLING}
EXC = {"STE": ExcitationMode.STE,
       "tEXC": ExcitationMode.TIME_DOMAIN_LAGRANGE,
       "fEXC": ExcitationMode.FREQUENCY_SAMPLING}

def _build(ks, exc):
    kw = dict(num_samples=NUM_SAMPLES, fs=FS, implementation=ks, excitation_mode=exc)
    if ks == Implementation.FREQUENCY_SAMPLING:
        kw["use_lti"] = True
    return Synth(SynthConfig(**kw)).to(DEVICE)

SYNTHS = {f"{k}.{e}": _build(kv, ev) for k, kv in KS.items() for e, ev in EXC.items()}
SW = SlicedWassersteinSpectralLoss(sample_rate=FS).to(DEVICE)

def make_params(**over):
    p = {"exists": torch.ones(1, 1, device=DEVICE),
         "time": torch.full((1, 1), GT["time"], device=DEVICE),
         "f0": torch.full((1, 1), GT["f0"], device=DEVICE),
         "burst_gain": torch.full((1, 1), GT["burst_gain"], device=DEVICE),
         "pluck_position": torch.full((1, 1), GT["pluck_position"], device=DEVICE),
         "dynamic_level": torch.full((1, 1), GT["dynamic_level"], device=DEVICE),
         "a1": torch.full((1, 1), GT["a1"], device=DEVICE),
         "decay": torch.full((1, 1), GT["decay"], device=DEVICE)}
    for k, v in over.items():
        p[k] = torch.tensor([[float(v)]], device=DEVICE)
    return p

# Oracle target (NumPy physical model).
with torch.no_grad():
    TARGET, _ = SYNTHS["tKSA.STE"].oracle_synth(make_params())
print("target:", tuple(TARGET.shape), "peak", float(TARGET.abs().max()))

def sw_loss(synth, **over):
    y, _ = synth(make_params(**over))
    return SW(y, TARGET)


---
## 1 · Finite-difference gradient check

For each config and parameter, compare the autograd gradient to a central
finite difference at two step sizes. **Sign-agreement** (robust to float32 FD
noise) is the headline; the two step sizes expose multi-scale roughness.

Read it as: smooth controls (`decay`, `dynamic_level`) should show ~100%
agreement (the check works); `time`+`STE` *will* diverge by design; instability
of the numeric gradient across step sizes flags a rough/multi-scale landscape,
**not** a bug.

In [ ]:
def fd_check(synth, pname, vals, h):
    sa, ratio = [], []
    for v in vals:
        p = make_params(); p[pname] = torch.tensor([[float(v)]], device=DEVICE, requires_grad=True)
        y, _ = synth(p); L = SW(y, TARGET); L.backward(); ga = float(p[pname].grad.item())
        with torch.no_grad():
            lp = sw_loss(synth, **{pname: v + h}).item()
            lm = sw_loss(synth, **{pname: v - h}).item()
        gn = (lp - lm) / (2 * h)
        sa.append(np.sign(ga) == np.sign(gn))
        ratio.append(abs(ga) / (abs(gn) + 1e-9))
    return float(np.mean(sa)), float(np.median(ratio))

PARAMS_FD = ["f0", "time", "decay", "dynamic_level"]
H = {"f0": (1.0, 0.2), "time": (2e-3, 5e-4), "decay": (1e-3, 2e-4), "dynamic_level": (5e-3, 1e-3)}
NPTS = 9

rows = []
for name, syn in tqdm(SYNTHS.items(), desc="FD check"):
    row = {"config": name}
    for pname in PARAMS_FD:
        lo, hi = PARAM_RANGES[pname]
        vals = np.linspace(lo + 0.15 * (hi - lo), hi - 0.15 * (hi - lo), NPTS)
        h1, h2 = H[pname]
        s1, _ = fd_check(syn, pname, vals, h1)
        s2, r2 = fd_check(syn, pname, vals, h2)
        row[f"{pname}:sign@h1"] = f"{s1:.0%}"
        row[f"{pname}:sign@h2"] = f"{s2:.0%}"
        row[f"{pname}:|ga/gn|"] = f"{r2:.0f}"
    rows.append(row)

df_fd = pd.DataFrame(rows).set_index("config")
print(df_fd.to_string())


---
## 2 · Oracle forward fidelity

Does each synth render the signal the oracle does? We compare the **onset
sample** (energy-envelope threshold crossing) at several requested onset times,
and the **spectral SNR** of each synth vs the oracle. A faithful forward keeps
the onset where the oracle puts it; a large onset error would legitimise a
misleading time gradient.

In [ ]:
def onset_sample(x):
    x = x.detach().squeeze().cpu().numpy()
    e = np.abs(x)
    k = 128
    e = np.convolve(e, np.ones(k) / k, mode="same")
    return int(np.argmax(e > 0.1 * e.max()))

def spectral_snr(x, ref):
    win = torch.hann_window(512, device=x.device)
    Sx = torch.stft(x.squeeze(0), 512, 256, window=win, return_complex=True).abs()
    Sr = torch.stft(ref.squeeze(0), 512, 256, window=win, return_complex=True).abs()
    num = (Sr ** 2).sum()
    den = ((Sr - Sx) ** 2).sum() + 1e-12
    return float(10 * torch.log10(num / den))

rows = []
for name, syn in tqdm(SYNTHS.items(), desc="forward fidelity"):
    for treq in [0.3, 0.5, 0.7]:
        p = make_params(time=treq)
        with torch.no_grad():
            o, _ = SYNTHS["tKSA.STE"].oracle_synth(p)
            y, _ = syn(p)
        rows.append({"config": name, "req_t": treq, "req_sample": int(treq * NUM_SAMPLES),
                     "oracle_onset": onset_sample(o), "synth_onset": onset_sample(y),
                     "onset_err": onset_sample(y) - onset_sample(o),
                     "spec_SNR_dB": round(spectral_snr(y, o), 1)})

df_fwd = pd.DataFrame(rows).set_index(["config", "req_t"])
print(df_fwd.to_string())


---
## 3 · Loss-landscape curves

Plot `L(θ)` over the full range. The **coarse** shape tells you whether descent
is guided to the target (global min at target ⇒ optimisable) regardless of the
fine-scale ripple that the local gradient sees. For `time` we add a **fine
zoom** to expose the sample-scale oscillation that makes the local gradient
noisy.

In [ ]:
def sweep(synth, pname, vals):
    with torch.no_grad():
        return np.array([sw_loss(synth, **{pname: v}).item() for v in vals])

def count_local_min(L):
    return int(np.sum((L[1:-1] < L[:-2]) & (L[1:-1] < L[2:])))

for pname in ["f0", "time"]:
    lo, hi = PARAM_RANGES[pname]
    vals = np.linspace(lo, hi, 81)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, exc in zip(axes, EXC):
        for ks in KS:
            cfg = f"{ks}.{exc}"
            L = sweep(SYNTHS[cfg], pname, vals)
            ax.plot(vals, L, label=f"{ks}  (#min={count_local_min(L)})")
        ax.axvline(GT[pname], color="k", ls="--", lw=1, label="target")
        ax.set_title(f"{pname} · {exc}")
        ax.set_xlabel(pname); ax.legend(fontsize=8)
    axes[0].set_ylabel("SW loss")
    fig.suptitle(f"Coarse loss landscape — {pname}")
    fig.tight_layout(); plt.show()

# Fine-scale zoom for time (±0.01 around target ~ ±640 samples).
t0 = GT["time"]
vals = np.linspace(t0 - 0.01, t0 + 0.01, 201)
fig, ax = plt.subplots(figsize=(9, 4))
for ks in KS:
    L = sweep(SYNTHS[f"{ks}.fEXC"], "time", vals)
    ax.plot(vals, L, label=f"{ks}.fEXC")
ax.axvline(t0, color="k", ls="--", lw=1, label="target")
ax.set_title("Fine-scale time landscape (sample-rate ripple)")
ax.set_xlabel("time"); ax.set_ylabel("SW loss"); ax.legend()
fig.tight_layout(); plt.show()


---
## Takeaways

Fill in after running, but the expected reading:

- **`time`-fKSA** — forward onset is correct (§2) and the coarse landscape's
  global min is at the target (§3), so the sub-chance CGA is **fine-scale
  gradient ripple**, not a bug. Mitigations: lower SW spectrogram resolution,
  more projections, or coarse-to-fine optimisation.
- **`f0`** — multiple local minima in §3 confirm genuine **pitch
  multimodality** (the harmonic-comb problem), not a backward error.
- **Controls** (`decay`, `dynamic_level`) pass the FD check (§1), confirming the
  diagnostics themselves are sound.
